## Reading data

In [2]:
import pandas as pd
import altair as alt

# Enables the transformer to allow more maximum rows
alt.data_transformers.enable("vegafusion")

# Read and rename columns appropriately, drop unnecessary block_amount
df = pd.read_csv("results.csv")
df = df.rename(columns={
    "max_a": "delta_a",
    "max_b": "delta_b",
    "block_size": "n/t",
    "r": "r"
})
df = df.drop(columns=["block_amount"])

# Rename algorithms
df["algorithm"] = df["algorithm"].replace({
    "Lis": "JKPT25",
    "Lars1": "JR",
    "Lars2": "Rohwedder25",
    "Gupta": "GJSZ25"
})

# Amount of entries before removing non-finished results
print(len(df))
df_finished = df[df["elapsed"].notna()]
# Amount of entries that actually finished processing
print(len(df_finished))
df_finished.head()

41933
5088


,algorithm,delta_a,delta_b,n/t,r,elapsed,error
2,Rohwedder25,1000,1000,2,1,86.0,none
10,Rohwedder25,1000,100,2,1,83.0,none
18,Rohwedder25,1000,10,2,1,75.0,none
20,GJSZ25,1000,10,2,1,2320.0,none
22,Rohwedder25,1000,10,4,1,9981.0,none


#### Overall comparison

In [3]:
chart = (
    alt.Chart(df_finished)
    .mark_boxplot(size=20)
    .encode(
        x=alt.X(
            "algorithm:N",
            title="Algorithm"
        ),
        y=alt.Y(
            "elapsed:Q",
            title="Time elapsed (ms)",
            scale=alt.Scale(
                type="log",
                domain=[df_finished["elapsed"].min(), 100_000_000]
            )
        ),
        color=alt.Color("algorithm:N", legend = None)
    )
    .properties(
        width=500,
        height=400
    )
)

display(chart)

chart.save("runtime_by_algorithm.pdf")

alt.Chart(...)

#### Benchmark values

In [4]:
benchmark_table = (
    df.groupby("algorithm")
      .agg(
          total_runs=("algorithm", "size"),
          successful_runs=("elapsed", "count"),
          median_runtime_ms=("elapsed", "median"),
          mean_runtime_ms=("elapsed", "mean"),
          min_runtime_ms=("elapsed", "min"),
          max_runtime_ms=("elapsed", "max"),
      )
      .reset_index()
)

benchmark_table["failed_runs"] = (
    benchmark_table["total_runs"] - benchmark_table["successful_runs"]
)

benchmark_table["success_rate"] = (
    benchmark_table["successful_runs"]
    / benchmark_table["total_runs"]
    * 100
)

benchmark_table = benchmark_table.drop(columns=["total_runs"])

# Reorder
benchmark_table = benchmark_table[
    [
        "algorithm",
        "median_runtime_ms",
        "mean_runtime_ms",
        "min_runtime_ms",
        "max_runtime_ms",
        "successful_runs",
        "failed_runs",
        "success_rate",
    ]
]

benchmark_table
table = benchmark_table[
    [
        "algorithm",
        "median_runtime_ms",
        "mean_runtime_ms",
        "success_rate",
    ]
]

print(table.to_latex(
    index=False,
    float_format="%.2f"
))

\begin{tabular}{lrrr}
\toprule
algorithm & median_runtime_ms & mean_runtime_ms & success_rate \\
\midrule
GJSZ25 & 1937.50 & 8904.28 & 8.49 \\
JKPT25 & 59716.00 & 1217371.52 & 7.79 \\
JR & 146.00 & 291548.25 & 4.98 \\
Rohwedder25 & 80.00 & 1489.38 & 27.59 \\
\bottomrule
\end{tabular}



#### Results grouped by a single attribute

In [5]:
inputs = ["delta_a", "delta_b", "n/t", "r"]

titles = {
    "delta_a": "Δ(A)",
    "delta_b": "Δ(B)",
    "n/t": "n and t",
    "r": "r"
}

for x in inputs:

    x_scale = (
        alt.Scale(type="log")
        if x in ["delta_a", "delta_b"]
        else alt.Scale(type="linear")
    )

    chart = alt.Chart(df_finished).mark_point().encode(
        x=alt.X(
            f"{x}:Q",
            title=titles[x],
            scale=x_scale
        ),
        y=alt.Y(
            "elapsed:Q",
            title="Time elapsed (ms)",
            scale=alt.Scale(type="log")
        ),
        color="algorithm:N",
    ).properties(
        title=f"Time elapsed based on {titles[x]}"
    )

titles = {
    "delta_a": "Δ(A)",
    "delta_b": "Δ(B)",
    "n/t": "n and t",
    "r": "r"
}

filenames = {
    "delta_a": "runtime_by_delta_a.pdf",
    "delta_b": "runtime_by_delta_b.pdf",
    "n/t": "runtime_by_n_t.pdf",
    "r": "runtime_by_r.pdf"
}

for x in inputs:
    x_scale = (
        alt.Scale(type="log")
        if x in ["delta_a", "delta_b"]
        else alt.Scale(type="linear")
    )

    chart = alt.Chart(df_finished).mark_point().encode(
        x=alt.X(
            f"{x}:Q",
            title=titles[x],
            scale=x_scale
        ),
        y=alt.Y(
            "elapsed:Q",
            title="Time elapsed (ms)",
            scale=alt.Scale(type="log")
        ),
        color=alt.Color("algorithm:N", legend=alt.Legend(title="Algorithm"))
    )

    chart.save(filenames[x])

    display(chart)

alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

#### Benchmark attributes

In [12]:
benchmark_table_delta_a = (
df.groupby(["algorithm", "delta_a"])
.agg(
total_runs=("algorithm", "size"),
successful_runs=("elapsed", "count"),
median_runtime_ms=("elapsed", "median"),
mean_runtime_ms=("elapsed", "mean"),
)
.reset_index()
)

benchmark_table_delta_a["success_rate"] = (
benchmark_table_delta_a["successful_runs"]
/ benchmark_table_delta_a["total_runs"]
* 100
)

benchmark_table_delta_a = benchmark_table_delta_a.drop(columns=["total_runs"])

# Remove rows with no successful runs / NaN runtime values

benchmark_table_delta_a = benchmark_table_delta_a.dropna(
subset=["median_runtime_ms", "mean_runtime_ms"]
)

table_delta_a = benchmark_table_delta_a[
[
"algorithm",
"delta_a",
"median_runtime_ms",
"mean_runtime_ms",
"success_rate",
]
]

print(table_delta_a.to_latex(
index=False,
float_format="%.2f"
))

table_delta_a

\begin{tabular}{lrrrr}
\toprule
algorithm & delta_a & median_runtime_ms & mean_runtime_ms & success_rate \\
\midrule
GJSZ25 & 1 & 839.00 & 2684.30 & 10.03 \\
GJSZ25 & 2 & 318.00 & 23047.38 & 8.90 \\
GJSZ25 & 3 & 566.00 & 8315.10 & 6.49 \\
GJSZ25 & 4 & 664.50 & 10444.31 & 6.50 \\
GJSZ25 & 5 & 2521.00 & 7654.54 & 8.20 \\
GJSZ25 & 6 & 544.00 & 2423.55 & 8.89 \\
GJSZ25 & 7 & 1972.00 & 2821.72 & 75.56 \\
GJSZ25 & 8 & 1760.00 & 4794.41 & 94.12 \\
GJSZ25 & 9 & 2345.00 & 5171.42 & 90.59 \\
GJSZ25 & 10 & 4322.50 & 7901.29 & 9.89 \\
GJSZ25 & 20 & 3624.50 & 6176.16 & 7.00 \\
GJSZ25 & 40 & 1955.00 & 3390.56 & 4.50 \\
GJSZ25 & 60 & 2345.00 & 2706.79 & 3.53 \\
GJSZ25 & 80 & 8464.00 & 25470.00 & 57.53 \\
GJSZ25 & 100 & 4731.00 & 12837.13 & 4.59 \\
GJSZ25 & 500 & 26081.00 & 45134.62 & 2.60 \\
GJSZ25 & 1000 & 2479.00 & 5765.57 & 1.71 \\
JKPT25 & 1 & 2874.00 & 1109838.55 & 12.89 \\
JKPT25 & 2 & 46860.00 & 1189677.53 & 10.33 \\
JKPT25 & 3 & 17499.50 & 2219152.71 & 9.51 \\
JKPT25 & 4 & 14090.50 & 1288907.

,algorithm,delta_a,median_runtime_ms,mean_runtime_ms,success_rate
0,GJSZ25,1,839.0,2684.300000,10.033445
1,GJSZ25,2,318.0,23047.383333,8.902077
2,GJSZ25,3,566.0,8315.102041,6.490066
3,GJSZ25,4,664.5,10444.312500,6.495264
4,GJSZ25,5,2521.0,7654.540984,8.198925
...,...,...,...,...,...
63,Rohwedder25,60,76.0,163.276243,22.824716
64,Rohwedder25,80,2882.0,7671.100000,82.191781
65,Rohwedder25,100,78.0,695.330000,24.125452
66,Rohwedder25,500,77.0,2067.237838,22.896040


In [ ]:
benchmark_table_delta_b = (
df.groupby(["algorithm", "delta_b"])
.agg(
total_runs=("algorithm", "size"),
successful_runs=("elapsed", "count"),
median_runtime_ms=("elapsed", "median"),
mean_runtime_ms=("elapsed", "mean"),
)
.reset_index()
)

benchmark_table_delta_b["success_rate"] = (
benchmark_table_delta_b["successful_runs"]
/ benchmark_table_delta_b["total_runs"]
* 100
)

benchmark_table_delta_b = benchmark_table_delta_b.drop(columns=["total_runs"])

# Remove rows with no successful runs / NaN runtime values

benchmark_table_delta_b = benchmark_table_delta_b.dropna(
subset=["median_runtime_ms", "mean_runtime_ms"]
)

table_delta_b = benchmark_table_delta_b[
[
"algorithm",
"delta_b",
"median_runtime_ms",
"mean_runtime_ms",
"success_rate",
]
]

print(table_delta_b.to_latex(
index=False,
float_format="%.2f"
))


In [7]:
benchmark_table_r = (
    df.groupby(["algorithm", "r"])
      .agg(
          total_runs=("algorithm", "size"),
          successful_runs=("elapsed", "count"),
          median_runtime_ms=("elapsed", "median"),
          mean_runtime_ms=("elapsed", "mean"),
      )
      .reset_index()
)

benchmark_table_r["success_rate"] = (
    benchmark_table_r["successful_runs"]
    / benchmark_table_r["total_runs"]
    * 100
)

benchmark_table_r = benchmark_table_r.drop(columns=["total_runs"])

# Remove rows with no successful runs / NaN runtime values
benchmark_table_r = benchmark_table_r.dropna(
    subset=["median_runtime_ms", "mean_runtime_ms"]
)

table_r = benchmark_table_r[
    [
        "algorithm",
        "r",
        "median_runtime_ms",
        "mean_runtime_ms",
        "success_rate",
    ]
]

print(table_r.to_latex(
    index=False,
    float_format="%.2f"
))

\begin{tabular}{lrrrr}
\toprule
algorithm & r & median_runtime_ms & mean_runtime_ms & success_rate \\
\midrule
GJSZ25 & 1 & 1719.00 & 7004.67 & 82.62 \\
GJSZ25 & 2 & 4789.00 & 22144.57 & 15.61 \\
GJSZ25 & 3 & 4769.00 & 5361.55 & 1.11 \\
GJSZ25 & 4 & 86.00 & 97.25 & 0.78 \\
GJSZ25 & 5 & 103.00 & 251.71 & 1.54 \\
GJSZ25 & 7 & 104.00 & 104.00 & 0.09 \\
GJSZ25 & 9 & 88.00 & 88.00 & 0.09 \\
JKPT25 & 1 & 32134.00 & 767958.86 & 56.56 \\
JKPT25 & 2 & 424870.50 & 2322605.84 & 17.56 \\
JKPT25 & 3 & 115323.50 & 2799930.02 & 3.80 \\
JR & 1 & 108.00 & 131678.19 & 30.57 \\
JR & 2 & 160.00 & 855096.63 & 16.98 \\
JR & 3 & 269.00 & 536.46 & 7.04 \\
JR & 4 & 394.00 & 890.33 & 2.06 \\
JR & 5 & 250.00 & 584.67 & 1.36 \\
JR & 6 & 1790.50 & 2761.17 & 0.54 \\
JR & 7 & 4954.00 & 4954.00 & 0.09 \\
JR & 8 & 41.00 & 41.00 & 0.09 \\
Rohwedder25 & 1 & 592.00 & 2396.89 & 91.67 \\
Rohwedder25 & 2 & 94.00 & 7050.64 & 37.82 \\
Rohwedder25 & 3 & 74.00 & 278.54 & 22.22 \\
Rohwedder25 & 4 & 73.00 & 108.95 & 20.00 \\
Rohw

#### All Algorithms compared without grouping (except r)

In [8]:
y_min = df_finished["elapsed"].min()
y_max = 100_000_000

for r_range, filename in [
    ([1, 2, 3, 4, 5], "runtime_by_r_1_5.pdf"),
    ([6, 7, 8, 9, 10], "runtime_by_r_6_10.pdf")
]:

    data = df_finished[df_finished["r"].isin(r_range)]

    chart = (
        alt.Chart(data)
        .mark_boxplot(size=15)
        .encode(
            x=alt.X(
                "algorithm:N",
                title="Algorithm"
            ),
            y=alt.Y(
                "elapsed:Q",
                title="Time elapsed (ms)",
                scale=alt.Scale(
                    type="log",
                    domain=[y_min, y_max]
                )
            ),
            color=alt.Color(
                "algorithm:N",
                legend=None
            ),
            column=alt.Column(
                "r:O",
                title="r"
            )
        )
        .properties(
            width=120,
            height=400
        )
    )

    chart.save(f"{filename}")
    display(chart)

alt.Chart(...)

alt.Chart(...)

#### Errors

In [9]:
error_stats = (
    df.groupby("r")["error"]
      .value_counts(normalize=True)
      .reset_index(name="percentage")
)

error_stats["percentage"] *= 100

error_chart = (
    alt.Chart(error_stats)
    .mark_bar()
    .encode(
        x=alt.X("r:O", title="r"),
        y=alt.Y(
            "percentage:Q",
            title="Percent of all runs",
            scale=alt.Scale(domain=[0, 100])
        ),
        color=alt.Color(
            "error:N",
            title="Error",
            scale=alt.Scale(
                domain=["none", "memory", "timeout", "encoding"],
                range=["#71DF67", "#5B8AF0", "#F7A233", "#AC2ED3"]
            )
        ),
        tooltip=[
            "r",
            "error",
            alt.Tooltip("percentage:Q", title="Percent", format=".1f")
        ]
    )
    .properties(
        title="Error distribution by r"
    )
)
display(error_chart)
error_chart.save('error_distribution.pdf')

alt.Chart(...)

In [10]:
error_stats = (
    df.groupby(["algorithm", "r"])["error"]
      .value_counts(normalize=True)
      .reset_index(name="percentage")
)

error_stats["percentage"] *= 100

algorithms = df["algorithm"].unique()

filenames = ["error_distribution_lis.pdf","error_distribution_lars1.pdf",
                "error_distribution_lars2.pdf", "error_distribution_gupta.pdf"]

for algorithm, filename in zip(algorithms, filenames):
    algorithm_stats = error_stats[
        error_stats["algorithm"] == algorithm
    ]

    error_chart = (
        alt.Chart(algorithm_stats)
        .mark_bar()
        .encode(
            x=alt.X("r:O", title="r"),
            y=alt.Y(
                "percentage:Q",
                title="Percent of runs",
                scale=alt.Scale(domain=[0, 100])
            ),
            color=alt.Color(
                "error:N",
                title="Error",
                scale=alt.Scale(
                    domain=["none", "memory", "timeout", "encoding"],
                    range=["#71DF67", "#5B8AF0", "#F7A233", "#AC2ED3"]
                )
            ),
            tooltip=[
                "r",
                "error",
                alt.Tooltip(
                    "percentage:Q",
                    title="Percent",
                    format=".1f"
                )
            ]
        )
        .properties(
            title=f"Error distribution by r — {algorithm}"
        )
    )

    display(error_chart)
    error_chart.save(f"{filename}")

alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

alt.Chart(...)